In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import random




In [2]:
train = pd.read_csv("../input/petfinder-pawpularity-score/train.csv")
test = pd.read_csv("../input/petfinder-pawpularity-score/test.csv")
sample = pd.read_csv("../input/petfinder-pawpularity-score/sample_submission.csv")




In [3]:
def fix_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


SEED = 42
fix_seed(SEED)




In [4]:
target = "Pawpularity"
features = train.columns.tolist()[1:-1]  # exclude Id and target column
print(f"Using {len(features)} features.")




Using 12 features.


In [5]:
folds = train.copy()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold_number, (_, val_idx) in enumerate(skf.split(folds, folds[target])):
    folds.loc[val_idx, "fold"] = fold_number
folds["fold"] = folds["fold"].astype(int)




/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


In [6]:
lgbm_params = {
    "objective": "rmse",
    "seed": SEED,
    "metric": "rmse",
    "learning_rate": 0.01,
    "max_bin": 800,
    "num_leaves": 80,
    "verbose": -1,
}




In [7]:
scores = []
all_preds = []
all_valid = pd.DataFrame()

for fold in range(5):
    fix_seed(SEED)  # ensure reproducibility per fold

    train_idx = folds[folds["fold"] != fold]
    val_idx = folds[folds["fold"] == fold]

    lgb_train = lgb.Dataset(train_idx[features], label=train_idx[target])
    lgb_valid = lgb.Dataset(val_idx[features], label=val_idx[target])

    # LightGBM training – use callbacks for early stopping and logging
    model = lgb.train(
        lgbm_params,
        lgb_train,
        num_boost_round=1000,
        valid_sets=[lgb_valid],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=50),
        ],
    )

    oof_pred = model.predict(val_idx[features])
    scores.append(np.sqrt(mean_squared_error(val_idx[target], oof_pred)))

    val_idx = val_idx.copy()
    val_idx["preds"] = oof_pred
    all_valid = pd.concat([all_valid, val_idx], ignore_index=True)

    test_pred = model.predict(test[features])
    all_preds.append(test_pred)




[50]	valid_0's rmse: 20.6222
[100]	valid_0's rmse: 20.6282
[50]	valid_0's rmse: 20.7245
[100]	valid_0's rmse: 20.7545
[50]	valid_0's rmse: 20.7009
[100]	valid_0's rmse: 20.6889
[150]	valid_0's rmse: 20.6827
[200]	valid_0's rmse: 20.6818
[250]	valid_0's rmse: 20.6853
[50]	valid_0's rmse: 20.5826
[100]	valid_0's rmse: 20.5913
[150]	valid_0's rmse: 20.6103
[50]	valid_0's rmse: 20.5534
[100]	valid_0's rmse: 20.5764


In [8]:
overall_rmse = np.mean(scores)
print(f"CV RMSE (mean of 5 folds): {overall_rmse:.5f}")




CV RMSE (mean of 5 folds): 20.62767


In [9]:
test_pred_mean = np.mean(all_preds, axis=0)
test_pred_clipped = np.clip(test_pred_mean, 1, 100)




In [10]:
sample["Pawpularity"] = test_pred_clipped
submission_path = "submission.csv"
sample.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Submission saved to submission.csv
